In [1]:
import torch
import torch.nn.functional as F

# =====================================================================
# STEP 1: CONSTRUCT A CAUSAL MASKED SELF-ATTENTION MODULE
# =====================================================================
def causal_masked_attention(Q, K, V):
    seq_len = Q.size(-2)
    d_k = Q.size(-1)
    
    # Compute base scores: (Batch, Seq_Len, Seq_Len)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    
    # Build a standard lower-triangular matrix of 1s
    # upper-triangular elements are filled with 0s
    mask = torch.tril(torch.ones(seq_len, seq_len))
    
    # Apply mask: Replace 0s with -inf to completely block future visibility
    scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Softmax turns -inf elements to 0.0 attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    output = torch.matmul(attention_weights, V)
    return output, attention_weights

# Verify masking properties (4 tokens, 8 dimensions)
mock_Q = torch.randn(1, 4, 8)
_, weights = causal_masked_attention(mock_Q, mock_Q, mock_Q)
print("--- Causal Attention Verification Matrix ---")
print(weights[0].round(decimals=3))
print("(Notice how the upper right triangle is completely filled with 0.0)\n")

# =====================================================================
# STEP 2: SIMULATE AN AUTOREGRESSIVE TOKEN GENERATION LOOP
# =====================================================================
class MockGPTDecoder:
    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        
    def forward(self, input_ids):
        # Simulating a decoder layer: returns random logits of shape (Batch, Seq_Len, Vocab_Size)
        batch_size, seq_len = input_ids.shape
        return torch.randn(batch_size, seq_len, self.vocab_size)

# Seed prompt initialized as a list of token IDs
input_tokens = torch.tensor([[12, 45, 87]]) # Batch size = 1, initial sequence length = 3
gpt_engine = MockGPTDecoder(vocab_size=100)

print("--- Launching Autoregressive Inference Loop ---")
for step in range(4):
    # 1. Run the sequence through the decoder
    logits = gpt_engine.forward(input_tokens)
    
    # 2. Extract ONLY the logit predictions for the final token position in the sequence
    next_token_logits = logits[:, -1, :]
    
    # 3. Apply Temperature scaling and convert to probabilities
    probabilities = F.softmax(next_token_logits / 0.7, dim=-1)
    
    # 4. Sample the next token ID from the probability distribution
    next_token_id = torch.multinomial(probabilities, num_samples=1)
    
    # 5. Append the newly generated token directly onto our tracking sequence array
    input_tokens = torch.cat([input_tokens, next_token_id], dim=1)
    
    print(f"Step {step+1} Generated Token ID: {next_token_id.item()} | Extended Sequence: {input_tokens[0].tolist()}")

--- Causal Attention Verification Matrix ---
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.0170, 0.9830, 0.0000, 0.0000],
        [0.0630, 0.0080, 0.9290, 0.0000],
        [0.0020, 0.0270, 0.0030, 0.9670]])
(Notice how the upper right triangle is completely filled with 0.0)

--- Launching Autoregressive Inference Loop ---
Step 1 Generated Token ID: 40 | Extended Sequence: [12, 45, 87, 40]
Step 2 Generated Token ID: 18 | Extended Sequence: [12, 45, 87, 40, 18]
Step 3 Generated Token ID: 29 | Extended Sequence: [12, 45, 87, 40, 18, 29]
Step 4 Generated Token ID: 83 | Extended Sequence: [12, 45, 87, 40, 18, 29, 83]
